In [1]:
# Import pandas for working with tabular data
import pandas as pd


In [2]:
# Load the spam dataset from the local CSV file
messages = pd.read_csv('../datasets/spam.csv', encoding='latin-1')
# Display the first few rows to inspect the data
messages.head()


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


### Data Cleaning and Preprocessing
# Clean the raw text data before building the Bag-of-Words model


In [3]:
# Check the dataset structure and column types
messages.info()


<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   v1          5572 non-null   str  
 1   v2          5572 non-null   str  
 2   Unnamed: 2  50 non-null     str  
 3   Unnamed: 3  12 non-null     str  
 4   Unnamed: 4  6 non-null      str  
dtypes: str(5)
memory usage: 217.8 KB


In [4]:
# Remove any columns that contain only missing values
messages = messages.dropna(axis=1)
# Confirm the cleaned dataset shape and remaining columns
messages.info()


<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   v1      5572 non-null   str  
 1   v2      5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB


In [5]:
# Rename the original columns to clearer names for model training
messages = messages.rename(columns={'v1': 'label', 'v2': 'message'})


In [6]:
# Preview the renamed dataset to verify the cleaned columns
messages.head()


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [7]:
# Import regular expressions and NLP tools needed for text cleaning
import re
import nltk
# Download the English stopwords list used to remove common filler words
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to /home/vscode/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
# Load stopwords and the stemming algorithm for text normalization
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer


In [9]:
# Create a stemmer to reduce words to their root form
stemmer = PorterStemmer()


In [10]:
# Clean each message, remove stopwords, and stem the remaining words
corpus = []

for i in range(0, len(messages)):
    # Keep only letters and replace everything else with spaces
    review = re.sub('[^a-zA-z]', ' ', messages['message'][i])
    review = review.lower()
    review = review.split()
    # Remove stopwords and reduce words to their root form
    review = [stemmer.stem(word) for word in review if word not in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)


In [ ]:
# Convert the cleaned text into a bag-of-words matrix
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=2500)


In [ ]:
# Build the feature matrix where each row is a message and each column is a word count
X = cv.fit_transform(corpus).toarray()
X


array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(5572, 2500))

In [ ]:
# Check the shape of the feature matrix: rows = messages, columns = unique words
X.shape


(5572, 2500)

In [ ]:
# View the vocabulary mapping from words to column indices
cv.vocabulary_


{'go': np.int64(804),
 'point': np.int64(1555),
 'crazi': np.int64(446),
 'avail': np.int64(141),
 'bugi': np.int64(274),
 'great': np.int64(828),
 'world': np.int64(2439),
 'la': np.int64(1079),
 'cine': np.int64(366),
 'got': np.int64(819),
 'wat': np.int64(2350),
 'ok': np.int64(1450),
 'lar': np.int64(1091),
 'joke': np.int64(1024),
 'wif': np.int64(2403),
 'oni': np.int64(1458),
 'free': np.int64(744),
 'entri': np.int64(614),
 'wkli': np.int64(2428),
 'comp': np.int64(401),
 'win': np.int64(2409),
 'fa': np.int64(657),
 'cup': np.int64(460),
 'final': np.int64(700),
 'tkt': np.int64(2183),
 'st': np.int64(1964),
 'may': np.int64(1269),
 'text': np.int64(2124),
 'receiv': np.int64(1664),
 'question': np.int64(1626),
 'std': np.int64(1980),
 'txt': np.int64(2248),
 'rate': np.int64(1645),
 'appli': np.int64(99),
 'dun': np.int64(575),
 'say': np.int64(1756),
 'earli': np.int64(579),
 'alreadi': np.int64(67),
 'nah': np.int64(1387),
 'think': np.int64(2147),
 'goe': np.int64(807),
 

In [29]:
# Create a second vectorizer with binary counts and n-grams for comparison
cv2 = CountVectorizer(max_features=200, binary=True, ngram_range=(2, 2))
# Fit the new vectorizer on the cleaned corpus before accessing its vocabulary
X2 = cv2.fit_transform(corpus).toarray()

In [30]:
# Now the vocabulary is available because cv2 was fitted above
cv2.vocabulary_

{'free entri': np.int64(54),
 'rate appli': np.int64(134),
 'claim call': np.int64(28),
 'call claim': np.int64(8),
 'claim code': np.int64(29),
 'free call': np.int64(53),
 'call mobil': np.int64(16),
 'chanc win': np.int64(27),
 'txt word': np.int64(177),
 'let know': np.int64(93),
 'feel like': np.int64(51),
 'repli ye': np.int64(140),
 'go home': np.int64(63),
 'call repli': np.int64(20),
 'mobil free': np.int64(105),
 'pleas call': np.int64(126),
 'lt gt': np.int64(100),
 'miss call': np.int64(103),
 'want go': np.int64(191),
 'first time': np.int64(52),
 'like lt': np.int64(94),
 'sm ac': np.int64(152),
 'sorri call': np.int64(153),
 'call later': np.int64(14),
 'award bonu': np.int64(5),
 'prize call': np.int64(131),
 'ur award': np.int64(180),
 'call free': np.int64(10),
 'that cool': np.int64(167),
 'call custom': np.int64(9),
 'custom servic': np.int64(39),
 'servic repres': np.int64(149),
 'guarante cash': np.int64(75),
 'cash prize': np.int64(26),
 'tri contact': np.int64(1

In [31]:
X2

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(5572, 200))